# AnnNet showcase

A guided tour of the public API. Ten sections, each self-contained, taking you from
the simplest binary graph to the multilayer math API.

This notebook is shipped with empty outputs — run all cells to populate.

**Sections**

1. Hello AnnNet
2. Attributes everywhere
3. Slices for conditions
4. Hyperedges for reactions
5. Multilayer for time / condition
6. Multilayer math
7. Views & subgraphs
8. IO buffet (round-trip via several formats)
9. Backend interop
10. History & replay


In [1]:
import annnet

print(annnet.__version__ if hasattr(annnet, '__version__') else 'dev')

0.2.0


## 1. Hello AnnNet

The simplest entry point. Construct a directed graph, add a few vertices and edges, look at the shape.


In [2]:
from annnet import AnnNet

G = AnnNet(directed=True)
G.add_vertices(['A', 'B', 'C', 'D'])
G.add_edges('A', 'B', edge_id='e1', weight=1.0)
G.add_edges('B', 'C', edge_id='e2', weight=2.0)
G.add_edges('C', 'D', edge_id='e3', weight=1.5)
G

AnnNet object with n_vertices × n_edges = 4 × 3
    directed: True
    slices: ['default']

In [3]:
# Shape
print('nv, ne =', G.nv, G.ne)
print('shape =', G.shape)
print('directed =', G.directed)

nv, ne = 4 3
shape = (4, 3)
directed = True


In [4]:
# The variadic add_vertices forms
G2 = AnnNet(directed=False)
G2.add_vertices('single')  # singleton string
G2.add_vertices(['a', 'b', 'c'])  # list of ids
G2.add_vertices([{'vertex_id': 'd', 'color': 'red'}])  # dict with attrs
G2.add_vertices([('e', {'color': 'blue'})])  # (id, attrs) tuple
print(sorted(G2.vertices()))

['a', 'b', 'c', 'd', 'e', 'single']


In [5]:
# Iterate edges
for eid in G._col_to_edge.values():
    rec = G._edges[eid]
    print(f'  {eid}: {rec.src} -> {rec.tgt}  weight={rec.weight}')

  e1: A -> B  weight=1.0
  e2: B -> C  weight=2.0
  e3: C -> D  weight=1.5


## 2. Attributes everywhere

Vertex, edge, slice, and graph-level attributes live in narwhals-backed tables.
Bulk variants exist for every setter.


In [6]:
# Vertex attributes
G.attrs.set_vertex_attrs('A', label='alpha', tier=1)
G.attrs.set_vertex_attrs_bulk(
    {
        'B': {'label': 'beta', 'tier': 1},
        'C': {'label': 'gamma', 'tier': 2},
        'D': {'label': 'delta', 'tier': 2},
    }
)

# Single-attr read
print('A.label =', G.attrs.get_attr_vertex('A', 'label'))
print('B full =', G.attrs.get_vertex_attrs('B'))

A.label = alpha
B full = {'label': 'beta', 'tier': 1}


In [7]:
# Edge attributes (bulk). 'kind' / 'weight' / 'directed' are reserved
# (structural columns), so use a different key.
G.attrs.set_edge_attrs_bulk(
    {
        'e1': {'confidence': 0.9, 'interaction': 'activation'},
        'e2': {'confidence': 0.7, 'interaction': 'inhibition'},
        'e3': {'confidence': 0.95, 'interaction': 'activation'},
    }
)

# Filter by attribute
activations = G.attrs.get_edges_by_attr('interaction', 'activation')
print('activation edges:', activations)

activation edges: ['e1', 'e3']


In [8]:
# Graph-level metadata (uns-style)
G.uns['study'] = 'showcase'
G.uns['author'] = 'demo'
print(G.attrs.get_graph_attributes())

{'study': 'showcase', 'author': 'demo'}


In [9]:
# Reserved keys are rejected — they're part of the structural contract
try:
    G.attrs.set_edge_attrs('e1', source='X')
except ValueError as e:
    print('caught:', e)

caught: edge attributes use reserved key(s): ['source']. These names are part of the structural / dispatch contract; rename your attribute(s) to use a different key.


## 3. Slices for conditions

Slices are condition / cohort / sample subsets — like obs categories in AnnData, but for vertices and edges together.


In [10]:
G.slices.add('control', cohort='wt')
G.slices.add('treated', cohort='ko')
print(G.slices.list())

['default', 'control', 'treated']


In [11]:
# Add edges directly into a slice
G.add_edges('A', 'D', edge_id='e_extra', slice='treated', weight=3.0)
print('treated edges:', G.slices.edges('treated'))
print('treated vertices:', G.slices.vertices('treated'))

treated edges: {'e_extra'}
treated vertices: {'D', 'A'}


In [12]:
# Slice algebra
G.slices.add_edges('control', ['e1', 'e2'])
G.slices.add_edges('treated', ['e2', 'e3', 'e_extra'])

both = G.slices.intersect(['control', 'treated'])
print('intersection (edges):', both['edges'])
union = G.slices.union(['control', 'treated'])
print('union (edges):', sorted(union['edges']))

intersection (edges): {'e2'}
union (edges): ['e1', 'e2', 'e3', 'e_extra']


In [13]:
# Per-slice weight overrides — useful for condition-specific weights
G.attrs.set_edge_slice_attrs('treated', 'e2', weight=99.0)
print('global e2 weight:', G.attrs.get_effective_edge_weight('e2'))
print('treated e2 weight:', G.attrs.get_effective_edge_weight('e2', slice='treated'))

global e2 weight: 2.0
treated e2 weight: 99.0


## 4. Hyperedges for reactions

A hyperedge connects an arbitrary set of vertices. Two flavours:

- **Undirected** — pass a list of members.
- **Directed** — pass `src=[heads]`, `tgt=[tails]`.


In [14]:
H = AnnNet(directed=False)
H.add_vertices(['Glc', 'ATP', 'G6P', 'ADP'])

# Undirected: a complex of three species
H.add_edges(['Glc', 'ATP', 'G6P'], edge_id='complex_A')

# Directed reaction: hexokinase  Glc + ATP -> G6P + ADP
H.add_edges(src=['Glc', 'ATP'], tgt=['G6P', 'ADP'], edge_id='hexokinase')
print('ne =', H.ne)
for eid in H._edges:
    rec = H._edges[eid]
    print(
        f'  {eid}: src={set(rec.src) if rec.src else None}, tgt={set(rec.tgt) if rec.tgt else None}'
    )

ne = 2
  complex_A: src={'ATP', 'Glc', 'G6P'}, tgt=None
  hexokinase: src={'ATP', 'Glc'}, tgt={'ADP', 'G6P'}


In [15]:
# Pretty edge view shows hyperedges with members / head / tail columns
H.views.edges().head(5)

edge_id,kind,source,target,edge_type,head,tail,members,directed,global_weight,effective_weight
str,str,str,str,null,list[str],list[str],list[str],bool,f64,f64
"""complex_A""","""hyper""","""ATP|G6P|Glc""",null,null,null,null,"[""ATP"", ""G6P"", ""Glc""]",false,1.0,1.0
"""hexokinase""","""hyper""","""ATP|Glc""","""ADP|G6P""",null,"[""ATP"", ""Glc""]","[""ADP"", ""G6P""]",null,true,1.0,1.0


In [16]:
# Stoichiometry: directly read/write the incidence matrix column
import numpy as np

M = H._matrix
col = H._edges['hexokinase'].col_idx
print('hexokinase column:')
for vid in H.vertices():
    rec_row = H._entities[H._resolve_entity_key(vid)].row_idx
    val = M[rec_row, col]
    print(f'  {vid:>6}: {val:+.2f}')

hexokinase column:
     Glc: +1.00
     ATP: +1.00
     G6P: -1.00
     ADP: -1.00


## 5. Multilayer for time / condition

Aspects partition vertices into layers. A vertex like `A` can live in multiple layers; the
*supra-node* `(A, ('t1',))` is the actual indexed entity. Intra / inter / coupling edges
have explicit roles.


In [17]:
import warnings

M = AnnNet(directed=False)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    # One aspect 't' with two elementary layers.
    M.layers.set_aspects(['t'], {'t': ['t1', 't2']})

    # Same vertices A and B in both layers.
    M.add_vertices(['A', 'B'], layer={'t': 't1'})
    M.add_vertices(['A', 'B'], layer={'t': 't2'})

    # Intra-layer edges.
    M.add_edges(('A', ('t1',)), ('B', ('t1',)), edge_id='e_t1', weight=1.0)
    M.add_edges(('A', ('t2',)), ('B', ('t2',)), edge_id='e_t2', weight=1.0)

print('aspects:', M.layers.list_aspects())
print('layers:', M.layers.list_layers())

aspects: ('t',)
layers: {'t': ['t1', 't2']}


In [18]:
# Add diagonal couplings (A in t1 <-> A in t2, B in t1 <-> B in t2)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    n_added = M.layers.add_layer_coupling_pairs([(('t1',), ('t2',))])
print('coupling edges added:', n_added)

coupling edges added: 2


In [19]:
# Subgraph for a specific layer
sub_t1 = M.layers.subgraph_from_layer_tuple(('t1',))
print('t1 subgraph: nv=', sub_t1.nv, 'ne=', sub_t1.ne)

t1 subgraph: nv= 2 ne= 1


## 6. Multilayer math

Supra-adjacency, supra-Laplacian, transition matrix, layer descriptors.
Same `M` graph from section 5.


In [20]:
A = M.layers.supra_adjacency()
print('supra-adjacency (', A.shape, '):')
print(A.toarray())

supra-adjacency ( (4, 4) ):
[[0. 1. 1. 0.]
 [1. 0. 0. 1.]
 [1. 0. 0. 1.]
 [0. 1. 1. 0.]]


In [21]:
# Decomposition: A = A_intra + A_inter + A_coup
A_intra = M.layers.build_intra_block().toarray()
A_inter = M.layers.build_inter_block().toarray()
A_coup = M.layers.build_coupling_block().toarray()
print('intra + inter + coupling == A?', np.allclose(A_intra + A_inter + A_coup, A.toarray()))

intra + inter + coupling == A? True


In [22]:
# Combinatorial Laplacian
L = M.layers.supra_laplacian(kind='comb').toarray()
print('L = D - A:')
print(L)
print('L symmetric?', np.allclose(L, L.T))

L = D - A:
[[ 2. -1. -1.  0.]
 [-1.  2.  0. -1.]
 [-1.  0.  2. -1.]
 [ 0. -1. -1.  2.]]
L symmetric? True


In [23]:
# Random walk transition matrix is row-stochastic
P = M.layers.transition_matrix().toarray()
print('row sums of P:', P.sum(axis=1))

row sums of P: [1. 1. 1. 1.]


In [24]:
# Spectral probes
lam2, fiedler = M.layers.algebraic_connectivity()
print('algebraic connectivity (lambda_2):', lam2)
print('Fiedler vector:', fiedler)

algebraic connectivity (lambda_2): 1.9999999999999998
Fiedler vector: [ 0.69847778 -0.11013078  0.11013078 -0.69847778]


In [25]:
# Layer-aware descriptors
pc = M.layers.participation_coefficient()
vers = M.layers.versatility()
print('participation coefficient:', pc)
print('versatility:', vers)

participation coefficient: {'A': 0.5, 'B': 0.5}
versatility: {'A': 1.0, 'B': 1.0}


## 7. Views & subgraphs

`G.views.*` returns lazy, filterable dataframes. The `materialize()` path on a `GraphView`
gives you a concrete subgraph.


In [26]:
# Edge table view
G.views.edges().head(5)

edge_id,kind,source,target,edge_type,head,tail,members,directed,global_weight,confidence,interaction,effective_weight
str,str,str,str,str,null,null,null,bool,f64,f64,str,f64
"""e1""","""binary""","""A""","""B""","""binary""",null,null,null,true,1.0,0.9,"""activation""",1.0
"""e2""","""binary""","""B""","""C""","""binary""",null,null,null,true,2.0,0.7,"""inhibition""",2.0
"""e3""","""binary""","""C""","""D""","""binary""",null,null,null,true,1.5,0.95,"""activation""",1.5
"""e_extra""","""binary""","""A""","""D""","""binary""",null,null,null,true,3.0,null,null,3.0


In [27]:
# Vertex table view
G.views.vertices().head(5)

vertex_id,label,tier
str,str,i64
"""A""","""alpha""",1
"""B""","""beta""",1
"""C""","""gamma""",2
"""D""","""delta""",2


In [28]:
# Slice table view
G.views.slices()

slice_id,cohort
str,str
"""default""",null
"""control""","""wt"""
"""treated""","""ko"""


In [29]:
# Materialize a filtered subgraph
from annnet.core._Views import GraphView

v = GraphView(G, vertices={'A', 'B', 'C'}, edges={'e1', 'e2'})
sub = v.materialize()
print('subgraph nv, ne =', sub.nv, sub.ne)
print('subgraph vertices:', sorted(sub.vertices()))

subgraph nv, ne = 3 2
subgraph vertices: ['A', 'B', 'C']


## 8. IO buffet

Round-trip the graph through several formats. Each branch is independent; cherry-pick
what you need.


In [30]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp(prefix='annnet_showcase_'))
print('using temp dir:', tmp)

using temp dir: /tmp/annnet_showcase_h7suo_1n


In [31]:
# Parquet (lossless)
from annnet import to_parquet, from_parquet

to_parquet(G, tmp / 'graph_parquet')
G_back = from_parquet(tmp / 'graph_parquet')
print('parquet round-trip: nv=', G_back.nv, 'ne=', G_back.ne)

parquet round-trip: nv= 4 ne= 4


In [32]:
# JSON
from annnet import to_json, from_json

to_json(G, tmp / 'graph.json')
G_json = from_json(tmp / 'graph.json')
print('json round-trip: nv=', G_json.nv, 'ne=', G_json.ne)

json round-trip: nv= 4 ne= 4


In [33]:
# Native .annnet format (compressed, manifest-aware)
from annnet import read, write

write(G, tmp / 'graph.annnet')
G_native = read(tmp / 'graph.annnet')
print('.annnet round-trip: nv=', G_native.nv, 'ne=', G_native.ne)

.annnet round-trip: nv= 4 ne= 4


In [34]:
# CX2 (Cytoscape Exchange) — useful for visualization
from annnet import to_cx2, from_cx2

cx2 = to_cx2(G)
G_cx2 = from_cx2(cx2)
print('cx2 round-trip: nv=', G_cx2.nv, 'ne=', G_cx2.ne)

cx2 round-trip: nv= 4 ne= 3


In [35]:
# DataFrames (per-table)
from annnet import to_dataframes

dfs = to_dataframes(G)
print('tables exported:', list(dfs.keys()))

tables exported: ['nodes', 'edges', 'hyperedges', 'slices', 'slice_weights']


## 9. Backend interop

Three backends: NetworkX (`G.nx`), igraph (`G.ig`), graph-tool (`G.gt`, optional).
Each is exposed as an accessor that forwards a familiar API onto a snapshot of the graph.


In [36]:
# NetworkX accessor: get the underlying nx graph via .backend()
import networkx as nx

nxG = G.nx.backend()
print('nx graph type:', type(nxG).__name__)
print('degree(A):', nxG.degree('A'))
print('shortest_path A->D:', nx.shortest_path(nxG, 'A', 'D'))

nx graph type: MultiDiGraph
degree(A): 2
shortest_path A->D: ['A', 'D']


/mnt/c/Users/pc/desktop/annnet-remote/annnet/core/backend_accessors/_base.py:156: RuntimeWarning: AnnNet-NX conversion is lossy: multiple slices flattened into single NX graph.
  entry = dict(build())


In [37]:
# Peek at vertex IDs the accessor sees
print('peek vertices:', G.nx.peek_vertices(10))

peek vertices: ['A', 'B', 'C', 'D']


In [38]:
# igraph accessor (same shape)
igG_view = G.ig.backend()
print('ig graph type:', type(igG_view).__name__)
print('vcount:', igG_view.vcount(), 'ecount:', igG_view.ecount())

ig graph type: Graph
vcount: 4 ecount: 4


/mnt/c/Users/pc/desktop/annnet-remote/annnet/core/backend_accessors/_base.py:156: RuntimeWarning: AnnNet-igraph conversion is lossy: multiple slices flattened into single igraph graph.
  entry = dict(build())


In [39]:
# Export to igraph + manifest, then re-import losslessly
from annnet import to_igraph, from_igraph

ig_graph, manifest = to_igraph(G)
G_via_ig = from_igraph(ig_graph, manifest)
print('igraph round-trip: nv=', G_via_ig.nv, 'ne=', G_via_ig.ne)

igraph round-trip: nv= 4 ne= 4


## 10. History & replay

`G.history` records every mutating call. Useful for provenance and for replaying the
construction pipeline on a fresh graph.


In [40]:
# Inspect the call log — G.history() is callable and returns a list of events
log = G.history()
print('total mutating calls:', len(log))
for entry in log[:5]:
    print(
        ' ',
        entry.get('op'),
        '-',
        {k: v for k, v in entry.items() if k in ('args', 'kwargs', 'result')},
    )

total mutating calls: 5
  add_vertices - {'result': ['A', 'B', 'C', 'D']}
  add_edges - {'args': ['A', 'B'], 'kwargs': {'edge_id': 'e1', 'weight': 1.0}, 'result': 'e1'}
  add_edges - {'args': ['B', 'C'], 'kwargs': {'edge_id': 'e2', 'weight': 2.0}, 'result': 'e2'}
  add_edges - {'args': ['C', 'D'], 'kwargs': {'edge_id': 'e3', 'weight': 1.5}, 'result': 'e3'}
  add_edges - {'args': ['A', 'D'], 'kwargs': {'edge_id': 'e_extra', 'slice': 'treated', 'weight': 3.0}, 'result': 'e_extra'}


In [41]:
# Tag named snapshots for later diffing
G.history.snapshot('after_construction')
G.attrs.set_vertex_attrs('A', extra='just-added')
G.add_vertices(['E'])
G.history.snapshot('after_extra')

# Each snapshot captures vertex / edge / slice id sets
for snap in G.history.list_snapshots():
    print(snap['label'], '— version:', snap['version'], 'nv:', len(snap['vertex_ids']))

after_construction — version: 5 nv: 4
after_extra — version: 7 nv: 5


In [42]:
# Diff two snapshots: what changed between marks?
diff = G.history.diff('after_construction', 'after_extra')
print(diff.summary())

Diff: after_construction - after_extra

Vertices: +1 added, 0 removed
Edges: +0 added, 0 removed
slices: +0 added, 0 removed


---

That's the tour. Next stops:

- `notebooks/special/sp01_directed_hyperedges.ipynb` — hyperedge stoichiometry deep-dive
- `notebooks/special/sp02_multilayer_math.ipynb` — supra-Laplacian / random walks / modularity
- `notebooks/special/sp03_flexible_edge_orientation.ipynb` — edge direction policies
- `notebooks/use_cases/uc1..uc3` — real-data workflows
- `notebooks/benchmark.ipynb` — tunable speed/memory profiling
